# Labwork 4
This labwork concerns the lecture 4.

The final objective is to describe how each column of a table of measurements is distributed, and to rescale it: its **p1, median and p99 without sorting anything**, then a clip and a rescale. That is scikit-learn's `RobustScaler`, which the last cell of this notebook writes with your own classes.

It is not one kernel but six patterns, one per exercise:

| exercise | pattern | its job in the pipeline |
|:--|:--|:--|
| 1 | exclusive scan | how many values lie *below* each bin — the cumulative distribution |
| 2 | reverse inclusive scan | how many lie *at or above* — the survival function |
| 3 | map | a value to the bin it falls in, and the rescaling itself |
| 4 | scatter | moving rows around, here a transpose |
| 5 | histogram | counting the bins |
| 6 | **quantiles** | exercises 1, 2, 3 and 5 assembled |

Notice that a histogram has few bins — 256 here — and a prefix sum over 256 numbers is less arithmetic than the launch that carries it. Exercise 1 will not shine on one column. A table has many, each with its own scan, which is what makes it worth doing at all.

Run the following cell first. It reports what hardware you were given and stops the notebook if that is a CPU: the runtime is declared in this file, but should you land on a CPU anyway, go to **Runtime ▸ Change runtime type ▸ T4 GPU** and run it again.

In [ ]:
!nvidia-smi
!nvcc --version

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("numba") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"],
                   check=True)

import numba

try:
    # Recent numba keeps the CUDA target in a separate distribution; the one
    # Colab ships still has it built in. Ask, rather than assume either way.
    from numba import config, cuda
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba-cuda"],
                   check=True)
    from numba import config, cuda

config.CUDA_ENABLE_PYNVJITLINK = True

print(f"numba {numba.__version__}")

if not cuda.is_available():
    raise SystemExit(
        "No CUDA device: Runtime -> Change runtime type -> T4 GPU, "
        "then Runtime -> Run all. Nothing below this cell can run on a CPU."
    )
cuda.detect()

### Checking your work

The cell below defines two helpers, provided: there is nothing to fill in.

`verify(...)` compares your result with a NumPy reference and **raises** when
they disagree, naming the first index that differs. A red cell means a wrong
kernel, a green one a right one.

`benchmark(...)` times the GPU against NumPy over a range of sizes, after
warm-up runs that pay for the JIT compilation: timing the first launch measures
the compiler, not the kernel.

Each exercise is followed by its own **`testN`**, which runs your class and
hands the result to the oracle for exercise *N*. You may call `test1(...)`
again at any point, on any array you like, and it still tests exercise 1.

In [ ]:
# --- provided: the self-check and benchmark helpers --------------------------
import contextlib
import io
from time import perf_counter

import numpy as np
from numba import core, cuda

# The first exercises launch a single block on purpose, and numba warns about
# it every time. The warning is right in general and noise here.
core.config.CUDA_LOW_OCCUPANCY_WARNINGS = False


class CheckFailed(AssertionError):
    """Raised when a kernel disagrees with its CPU reference."""


# Named `verify` rather than `check` because each exercise gets its own
# `checkN`: that one is the oracle for exercise N, this one is the primitive
# they are all written in terms of.


def verify(name, got, expected, tolerance=0.0):
    """Compare a result against its oracle, and raise on the first difference.

    `tolerance` is relative; leave it at 0 for integers, where the answer is
    exact, and pass something like 1e-5 for floats.
    """
    got, expected = np.asarray(got), np.asarray(expected)
    if got.shape != expected.shape:
        raise CheckFailed(
            f"{name}: shape is {got.shape}, expected {expected.shape}"
        )
    if tolerance:
        wrong = ~np.isclose(got, expected, rtol=tolerance, atol=tolerance)
    else:
        wrong = got != expected
    count = int(np.count_nonzero(wrong))
    if count:
        first = int(np.argmax(wrong))
        raise CheckFailed(
            f"{name}: {count} of {got.size} values are wrong. "
            f"First at index {first}: got {got.ravel()[first]}, "
            f"expected {expected.ravel()[first]}."
        )
    print(f"PASS  {name}  ({got.size} value{'' if got.size == 1 else 's'})")


def _time_once(call):
    """One timed call, in milliseconds, with the device drained first.

    Most of these classes print their own launch sizes and timings on every
    call, which would bury the table below under a few hundred lines, so the
    timed call is run with its output swallowed.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        start = perf_counter()
        call()
        cuda.synchronize()
        stop = perf_counter()
    return (stop - start) * 1000.0


def peak_bandwidth():
    """The card's theoretical memory bandwidth in GB/s, or None if unreadable.

    Peak is what a reduction or a scan should be measured against. These
    kernels do almost no arithmetic: they read the array once and produce a
    handful of numbers, so the only question worth asking is how close to the
    memory system they get. A speed-up against one CPU core answers a much
    easier question.
    """
    try:
        device = cuda.get_current_device()
        clock_khz = device.MEMORY_CLOCK_RATE            # kHz
        bus_bytes = device.GLOBAL_MEMORY_BUS_WIDTH / 8  # bits -> bytes
        return 2 * clock_khz * 1e3 * bus_bytes / 1e9    # 2: DDR
    except Exception:
        return None


def benchmark(name, build, sizes, repeats=10, warmup=3):
    """Time the GPU against NumPy over `sizes`.

    `build(size)` returns `(gpu, cpu)`, or `(gpu, cpu, nbytes)` to get the
    bandwidth columns as well. The two callables take no arguments and already
    hold their data, so the upload is not timed unless the kernel needs it.
    `nbytes` is the traffic one call claims to move; what counts as traffic is
    said where each `build` is written, because it is not the same question for
    a reduction, a scan and a sort.
    """
    peak = peak_bandwidth()
    print(name + (f"   —   this card peaks at {peak:.0f} GB/s" if peak else ""))
    print(f"{'size':>12} | {'GPU (ms)':>9} | {'NumPy (ms)':>11} | "
          f"{'speed-up':>9} | {'GB/s':>7} | {'of peak':>8}")
    print("-" * 72)
    for size in sizes:
        with contextlib.redirect_stdout(io.StringIO()):
            built = build(size)
            gpu, cpu = built[0], built[1]
            nbytes = built[2] if len(built) > 2 else None
            for _ in range(warmup):
                gpu()
            cuda.synchronize()
        best = min(_time_once(gpu) for _ in range(repeats))
        reference = min(_time_once(cpu) for _ in range(3))

        rate = None if nbytes is None else nbytes / (best / 1000.0) / 1e9
        rate_text = "      -" if rate is None else f"{rate:7.1f}"
        share_text = ("       -" if rate is None or peak is None
                      else f"{100 * rate / peak:7.1f}%")
        print(f"{size:12d} | {best:9.3f} | {reference:11.3f} | "
              f"{reference / best:8.1f}x | {rate_text} | {share_text}")

## Exercise 1
Here we start with the simple question: write the exclusive scan (so labwork 3 with some basic modification)...

In [ ]:
from __future__ import annotations

from typing import Any, Callable

import numpy as np
from numba import cuda, core
from numba.cuda.cudadrv.devicearray import DeviceNDArray
from numba.cuda.cudadrv.driver import Stream
from numba.np.numpy_support import from_dtype


def display(array, all_values=False):
    def aprint(data_range, noend=False):
        for i in data_range:
            print(f"{array[i]} ", end="")
            if (i % 16) == 15: print("")
        if not noend: print("...")

    if all_values:
        aprint(range(array.size), noend=True)
        return
    aprint(range(0, min(array.size, 16)))
    if array.size < 256 - 16: return
    aprint(range(256 - 16, min(array.size, 256 + 16)))
    if array.size < 512 - 16: return
    aprint(range(512 - 16, min(array.size, 512 + 16)))
    if array.size < 768 - 16: return
    aprint(range(768 - 16, min(array.size, 768 + 16)))
    aprint(range(array.size - 16 - 256, array.size - 256 + 16))
    aprint(range(array.size - 16, array.size), noend=True)


class ExclusiveScan(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _kernels_block_scan = {}
    _kernels_block_map = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_block_scan_factory(fn, np_type):
        """Factory of kernels for the block-scan problem...

        This function returns a Cuda Kernel that does block-scan of some data using a given binary functor."""

        scan_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        max_block_size = ExclusiveScan._NUM_WARPS * ExclusiveScan._WARP_SIZE

        def gpu_scan_block(d_input, d_output, d_extras, null_value):
            """
            Per block SCAN...
            """
            # TODO

        return cuda.jit(func_or_sig=gpu_scan_block, cache=True)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile_block_scan(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_scan:
            self._kernels_block_scan[key] = \
                ExclusiveScan._gpu_kernel_block_scan_factory(self._functor, from_dtype(dtype))
        return self._kernels_block_scan[key]

    @staticmethod
    def _gpu_kernel_block_map_factory(fn, np_type):
        """Factory of kernels for the block-map problem...

        This function returns a Cuda Kernel that does block-map of some data using a given binary functor."""

        scan_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def gpu_map_block(d_io, d_scan):
            """
            Per block MAP...
            """
            # TODO: implements the logics
            tid = cuda.grid(1)
            bid = cuda.blockIdx.x
            if bid == 0:
                # skip the first block (no map needed)
                return
            extra = d_scan[bid]
            d_io[tid] = scan_op(extra, d_io[tid])

        return cuda.jit(func_or_sig=gpu_map_block, cache=True)

    def _compile_block_map(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_map:
            self._kernels_block_map[key] = \
                ExclusiveScan._gpu_kernel_block_map_factory(self._functor, from_dtype(dtype))
        return self._kernels_block_map[key]

    def __call__(self, d_input, d_output, null_value, stream=cuda.default_stream()):
        """ Performs a per-block SCAN.

        :param d_input: A device array with input data.
        :param d_output: A device array to store the result.
        :param null_value: The null value for the
        :param stream: Optional CUDA stream in which to perform the scan.
                    If no stream is specified, the default stream of 0 is used.
        """

        # ensure 1d array
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # ensure arrays' size are the same
        if d_input.size != d_output.size:
            raise ValueError("arrays' length are different ({d_input.size} / {d_output.size}")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel_step1 = self._compile_block_scan(d_input.dtype)
        kernel_step2 = self._compile_block_map(d_input.dtype)

        max_block_size = ExclusiveScan._WARP_SIZE * ExclusiveScan._NUM_WARPS
        nb_threads = min(max_block_size, d_input.size)
        nb_blocks = (d_input.size + nb_threads - 1) // nb_threads
        extras = cuda.device_array(shape=nb_blocks, dtype=d_input.dtype)

        # Perform the reduction on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        kernel_step1[nb_blocks, nb_threads, stream](d_input, d_output, extras, null_value)
        # print(f"launch kernel_step1[{nb_blocks}, {nb_threads}]")
        if nb_blocks > 1:
            # display(extras, all=True)
            self(extras, extras, null_value, stream)
            kernel_step2[nb_blocks, nb_threads, stream](d_output, extras)

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # display(extras)

        return cuda.event_elapsed_time(start_event, stop_event)


#### Tests — exercise 1 — the exclusive scan

In [ ]:
def test1(h_input, scan_class=ExclusiveScan):
    def check1(result):
        """The oracle is `cumsum(h) - h`, which is what the kernel computes: it
        writes `null_value` at lane 0 and `shared[local_tid - 1]` elsewhere.

        The handout compared against `concatenate((zeros(1), cumsum(h[1:])))`,
        which agrees with that only when every value is the same -- which is
        why testing with `np.ones` never caught the difference.
        """
        verify(f"exclusive scan of {h_input.size} values", result,
               np.cumsum(h_input) - h_input)

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    ct = scanner(cuda.to_device(h_input), d_output, h_input.dtype.type(0))
    check1(d_output.copy_to_host())
    return ct


rng = np.random.default_rng(1)
for size in (256, 257, 1 << 12, (1 << 16) + 13):
    test1(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

#### Benchmark — the exclusive scan

In [ ]:
def build(size):
    h_input = np.random.default_rng(size).integers(
        0, 1 << 10, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    d_output = cuda.device_array(shape=size, dtype=np.int32)
    scanner = ExclusiveScan(lambda a, b: a + b)
    zero = np.int32(0)
    # Read once, write once: twice the array.
    return ((lambda: scanner(d_input, d_output, zero)),
            (lambda: np.cumsum(h_input) - h_input), 2 * h_input.nbytes)


benchmark("exclusive scan: GPU against NumPy",
          build, (1 << 20, 1 << 22, 1 << 24))

## Exercise 2
Here we continue with the Inclusive scan in the reversed version (so, labwork 3 with some minor modifications again...).

In [ ]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core


def display(array, all_values=False):
    def aprint(data_range, noend=False):
        for i in data_range:
            print(f"{array[i]} ", end="")
            if (i % 16) == 15: print("")
        if not noend: print("...")

    if all_values:
        aprint(range(array.size), noend=True)
        return
    aprint(range(0, min(array.size, 16)))
    if array.size < 256 - 16: return
    aprint(range(256 - 16, min(array.size, 256 + 16)))
    if array.size < 512 - 16: return
    aprint(range(512 - 16, min(array.size, 512 + 16)))
    if array.size < 768 - 16: return
    aprint(range(768 - 16, min(array.size, 768 + 16)))
    aprint(range(array.size - 16 - 256, array.size - 256 + 16))
    aprint(range(array.size - 16, array.size), noend=True)


class ReverseIScan(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _kernels_block_scan = {}
    _kernels_block_map = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_block_scan_factory(fn, np_type):
        """Factory of kernels for the block-scan problem...

        This function returns a Cuda Kernel that does block-scan of some data using a given binary functor."""

        scan_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        max_block_size = ReverseIScan._NUM_WARPS * ReverseIScan._WARP_SIZE

        @cuda.jit(device=True, cache=True)
        def load_shared_memory(shared_memory, d_input):
            local_tid = cuda.threadIdx.x
            tid = cuda.grid(1)
            # TODO: load data into shared memory

        @cuda.jit(device=True, cache=True)
        def pointer_jumping(shared_memory, jump):
            tid = cuda.threadIdx.x
            # TODO: implement the pointer jumping

        @cuda.jit(device=True, cache=True)
        def save_shared_memory(shared_memory, d_output, d_extras):
            local_tid = cuda.threadIdx.x
            global_tid = cuda.grid(1)
            # TODO: save data from shared memory

        def gpu_scan_block(d_input, d_output, d_extras):
            """
            Per block SCAN...
            """
            # move data to shared memory
            shared_memory = cuda.shared.array(shape=max_block_size, dtype=np_type)
            load_shared_memory(shared_memory, d_input)

            # implements the logics
            jump = 1
            while jump < cuda.blockDim.x:
                pointer_jumping(shared_memory, jump)
                jump *= 2

            # now stores the result
            save_shared_memory(shared_memory, d_output, d_extras)

        return cuda.jit(func_or_sig=gpu_scan_block, cache=True)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile_block_scan(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_scan:
            self._kernels_block_scan[key] = \
                ReverseIScan._gpu_kernel_block_scan_factory(self._functor, from_dtype(dtype))
        return self._kernels_block_scan[key]

    @staticmethod
    def _gpu_kernel_block_map_factory(fn, np_type):
        """Factory of kernels for the block-map problem...

        This function returns a Cuda Kernel that does block-map of some data using a given binary functor."""

        scan_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def gpu_map_block(d_io, d_scan):
            """
            Per block MAP...
            """
            # TODO: implements the logics
            tid = cuda.grid(1)
            bid = cuda.blockIdx.x
            if bid + 1 == cuda.gridDim.x:
                # skip the last block (no map needed)
                return
            extra = d_scan[bid + 1]
            d_io[tid] = scan_op(extra, d_io[tid])

        return cuda.jit(func_or_sig=gpu_map_block, cache=True)

    def _compile_block_map(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_map:
            self._kernels_block_map[key] = \
                ReverseIScan._gpu_kernel_block_map_factory(self._functor, from_dtype(dtype))
        return self._kernels_block_map[key]

    def __call__(self, d_input, d_output, stream=cuda.default_stream()):
        """ Performs a per-block SCAN.

        :param d_input: A device array with input data.
        :param d_output: A device array to store the result.
        :param stream: Optional CUDA stream in which to perform the scan.
                    If no stream is specified, the default stream of 0 is used.
        """

        # ensure 1d array
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # ensure arrays' size are the same
        if d_input.size != d_output.size:
            raise ValueError("arrays' length are different ({d_input.size} / {d_output.size}")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel_step1 = self._compile_block_scan(d_input.dtype)
        kernel_step2 = self._compile_block_map(d_input.dtype)

        max_block_size = ReverseIScan._WARP_SIZE * ReverseIScan._NUM_WARPS
        nb_threads = min(max_block_size, d_input.size)
        nb_blocks = (d_input.size + nb_threads - 1) // nb_threads
        extras = cuda.device_array(shape=nb_blocks, dtype=d_input.dtype)

        # Perform the reduction on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        kernel_step1[nb_blocks, nb_threads, stream](d_input, d_output, extras)
        # print(f"launch kernel_step1[{nb_blocks}, {nb_threads}]")
        if nb_blocks > 1:
            # display(extras, all=True)
            self(extras, extras, stream)
            kernel_step2[nb_blocks, nb_threads, stream](d_output, extras)

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # display(extras)

        return cuda.event_elapsed_time(start_event, stop_event)


#### Tests — exercise 2 — the reverse inclusive scan

In [ ]:
def test2(h_input, scan_class=ReverseIScan):
    def check2(result):
        """Suffix sums: `pointer_jumping` accumulates `shared[tid + jump]` into
        `shared[tid]`, so position i ends holding `h[i:].sum()`.

        The handout's `flip(cumsum(h))` is a different array unless every value
        is the same.
        """
        verify(f"reverse inclusive scan of {h_input.size} values", result,
               np.flip(np.cumsum(np.flip(h_input))))

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    ct = scanner(cuda.to_device(h_input), d_output)
    check2(d_output.copy_to_host())
    return ct


rng = np.random.default_rng(2)
for size in (256, 257, 1 << 12, (1 << 16) + 13):
    test2(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

## Exercise 3
The quantiles need a Unary Map pattern: turning a value into the bin it falls in is a map, and turning a count into a probability is another one...
Fill the following classes to implement the kernel logics of unary and binary map.

In [ ]:
class UnaryMap:
    _kernels_cache = {}

    def __init__(self: UnaryMap, functor) -> None:
        self.__functor = functor

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the unary transform problem...

        This function returns a Cuda Kernel that does the unary map of some data using a given unary functor."""

        umap_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def kernel(d_input, d_output) -> None:
            tid = cuda.grid(1)
            # TODO

        return cuda.jit(func_or_sig=kernel, cache=True)

    @staticmethod
    def _compile(functor, dtype):

        key = functor, dtype
        if key not in UnaryMap._kernels_cache:
            UnaryMap._kernels_cache[key] = UnaryMap._gpu_kernel_factory(functor, from_dtype(dtype))

        return UnaryMap._kernels_cache[key]

    def __call__(self, arr, res=None, stream=0):
        """Performs a full unary transform.

        :param arr: A host or device array.
        :param res: Optional device array into which to write the transform
                    result to. The result is written into the first element of
                    this array. If this parameter is specified, then no
                    communication of the map output takes place from the
                    device to the host.
        :param stream: Optional CUDA stream in which to perform the transform.
                    If no stream is specified, the default stream of 0 is used.
        :return: If ``res`` is specified, ``None`` is returned. Otherwise, the
                result of the reduction is returned.
        """

        # ensure 1d array
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if arr.size < 1:
            raise ValueError("array's length is 0")

        kernel = self._compile(self.__functor, arr.dtype)

        # Perform the MAP on the GPU
        nb_threads = 512
        nb_blocks = (arr.size + nb_threads - 1) // nb_threads

        dest = res
        if dest is None:
            dest = cuda.device_array_like(arr)

        elif dest.size != arr.size:
            raise ValueError("destination array's length does not equal input one")

        # Timed like every other exercise of these labworks, so that a
        # map can be put beside the scans and reduces it is built from.
        start_event = cuda.event(True)
        stop_event = cuda.event(True)
        start_event.record(stream=stream)
        kernel[nb_blocks, nb_threads, stream](arr, dest)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        if res is not None:
            return None, ct

        return dest, ct


class BinaryMap:
    _kernels_cache = {}

    def __init__(self: BinaryMap, functor) -> None:
        self.__functor = functor

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the binary transform problem...

        This function returns a Cuda Kernel that does the binary map of some data using a given binary functor."""

        map_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def kernel(d_input_left, d_input_right, d_output) -> None:
            tid = cuda.grid(1)
            # TODO

        return cuda.jit(func_or_sig=kernel, cache=True)

    @staticmethod
    def _compile(functor, dtype):

        key = functor, dtype
        if key not in BinaryMap._kernels_cache:
            BinaryMap._kernels_cache[key] = BinaryMap._gpu_kernel_factory(functor, from_dtype(dtype))

        return BinaryMap._kernels_cache[key]

    def __call__(self, left, right, res=None, stream=0):
        """Performs a full binary transform.

        :param left: A host or device array, used as the left operand.
        :param right: A host or device array, used as the right operand.
        :param res: Optional device array into which to write the transform
                    result to. The result is written into the first element of
                    this array. If this parameter is specified, then no
                    communication of the map output takes place from the
                    device to the host.
        :param stream: Optional CUDA stream in which to perform the transform.
                    If no stream is specified, the default stream of 0 is used.
        :return: If ``res`` is specified, ``None`` is returned. Otherwise, the
                result of the reduction is returned.
        """

        # ensure 1d array
        if left.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if left.size < 1:
            raise ValueError("array's length is 0")

        # ensure same size
        if left.size != right.size:
            raise ValueError("input arrays' length are not equal")

        dest = res
        if dest is None:
            dest = cuda.device_array_like(left)

        elif dest.size != left.size:
            raise ValueError("destination array's length does not equal input ones")

        kernel = self._compile(self.__functor, left.dtype)

        # Perform the MAP on the GPU
        nb_threads = 512
        nb_blocks = (left.size + nb_threads - 1) // nb_threads

        # Timed like every other exercise of these labworks, so that a
        # map can be put beside the scans and reduces it is built from.
        start_event = cuda.event(True)
        stop_event = cuda.event(True)
        start_event.record(stream=stream)
        kernel[nb_blocks, nb_threads, stream](left, right, dest)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        if res is not None:
            return None, ct

        return dest, ct


#### Tests — exercise 3 — map

In [ ]:
def test3(h_input, h_right=None, unary=None, binary=None,
          unary_class=UnaryMap, binary_class=BinaryMap):
    """One map, unary or binary, checked against the same operation in NumPy."""
    def check3(name, expected, result):
        # The oracle for a map is the functor itself, applied by NumPy to the
        # whole array: if the two disagree anywhere, the kernel did not apply
        # it to that element.
        verify(name, result, expected)

    d_input = cuda.to_device(h_input)
    d_output = cuda.device_array_like(d_input)
    if h_right is None:
        _, ct = unary_class(unary)(d_input, d_output)
        check3("unary map", unary(h_input), d_output.copy_to_host())
    else:
        _, ct = binary_class(binary)(d_input, cuda.to_device(h_right), d_output)
        check3("binary map", binary(h_input, h_right), d_output.copy_to_host())
    return ct


rng = np.random.default_rng(3)
size = 1 << 16
test3(rng.integers(0, 1 << 10, size=size, dtype=np.int32),
      unary=lambda x: 2 * x + 1)
test3(rng.integers(0, 1 << 10, size=size, dtype=np.int32),
      h_right=rng.integers(0, 1 << 10, size=size, dtype=np.int32),
      binary=lambda l, r: l - r)

## Exercise 4

Reading a quantile off a cumulative distribution is a permutation: for each probability wanted you fetch the value sitting at the bin you found — a gather whose index had to be *searched for* rather than computed.
Fill the following class for the scatter... And since we are speaking about permutation, do also the gather for the same price :-)

Notice that the mapping is given through a functor, that can internaly use an array for instance or some private data...

In [ ]:
class Scatter:
    _kernels_cache = {}

    def __init__(self: Scatter, functor: Callable) -> None:
        self.__functor = functor

    @staticmethod
    def _gpu_kernel_factory(fn: Callable, np_type: Any) -> cuda.FakeCUDAKernel:
        """Factory of kernels for the scatter permutation...

        This function returns a Cuda Kernel that does the permutation of some data."""

        map_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def kernel(d_input, d_output) -> None:
            tid = cuda.grid(1)
            # TODO

        return cuda.jit(func_or_sig=kernel, cache=True)

    @staticmethod
    def _compile(functor: Callable, dtype: Any) -> cuda.FakeCUDAKernel:

        key = functor, dtype
        if key not in Scatter._kernels_cache:
            Scatter._kernels_cache[key] = Scatter._gpu_kernel_factory(functor, from_dtype(dtype))

        return Scatter._kernels_cache[key]

    def __call__(self, src: DeviceNDArray, dest: DeviceNDArray, stream: int = 0) -> float:
        """Performs a full scatter.

        :param src: A host or device array.
        :param dst: Device array into which to write the permuted data to.
        :param stream: Optional CUDA stream in which to perform the transform.
                    If no stream is specified, the default stream of 0 is used.
        """

        # ensure device array
        cuda.cudadrv.devicearray.require_cuda_ndarray(src)
        cuda.cudadrv.devicearray.require_cuda_ndarray(dest)

        # ensure 1d array
        if src.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if src.size < 1:
            raise ValueError("array's length is 0")

        # ensure same shape/size
        if (src.shape != dest.shape) or (src.size != dest.size):
            raise ValueError("source and destination arrays' length are different")

        kernel = self._compile(self.__functor, src.dtype)

        # Perform the SCATTER on the GPU
        nb_threads = 512
        nb_blocks = (src.size + nb_threads - 1) // nb_threads
        # Timed like every other exercise of these labworks. Returned,
        # never printed: the caller decides what to do with it.
        start_event = cuda.event(True)
        stop_event = cuda.event(True)
        start_event.record(stream=stream)
        kernel[nb_blocks, nb_threads, stream](src, dest)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        return cuda.event_elapsed_time(start_event, stop_event)


class Gather:
    _kernels_cache = {}

    def __init__(self: Gather, functor: Callable) -> None:
        self.__functor = functor

    @staticmethod
    def _gpu_kernel_factory(fn: Callable, np_type: Any) -> cuda.FakeCUDAKernel:
        """Factory of kernels for the gather permutation...

        This function returns a Cuda Kernel that does the permutation of some data."""

        map_op = cuda.jit(device=True)(fn)

        def kernel(d_input, d_output) -> None:
            tid = cuda.grid(1)
            # TODO

        return cuda.jit(func_or_sig=kernel, cache=True)

    @staticmethod
    def _compile(functor: Callable, dtype: Any) -> cuda.FakeCUDAKernel:

        key = functor, dtype
        if key not in Gather._kernels_cache:
            Gather._kernels_cache[key] = Gather._gpu_kernel_factory(functor, from_dtype(dtype))

        return Gather._kernels_cache[key]

    def __call__(self, src: DeviceNDArray, dest: DeviceNDArray, stream: int = 0) -> float:
        """Performs a full gather.

        :param src: A host or device array.
        :param dst: Device array into which to write the permuted data to.
        :param stream: Optional CUDA stream in which to perform the transform.
                    If no stream is specified, the default stream of 0 is used.
        """

        # ensure device array
        cuda.cudadrv.devicearray.require_cuda_ndarray(src)
        cuda.cudadrv.devicearray.require_cuda_ndarray(dest)

        # ensure 1d array
        if src.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if src.size < 1:
            raise ValueError("array's length is 0")

        # ensure same shape/size
        if (src.shape != dest.shape) or (src.size != dest.size):
            raise ValueError("arrays' share/size are not equal")

        kernel = self._compile(self.__functor, src.dtype)

        # Perform the gather on the GPU
        nb_threads = 512
        nb_blocks = (src.size + nb_threads - 1) // nb_threads

        # Timed like every other exercise of these labworks. Returned,
        # never printed: the caller decides what to do with it.
        start_event = cuda.event(True)
        stop_event = cuda.event(True)
        start_event.record(stream=stream)
        kernel[nb_blocks, nb_threads, stream](src, dest)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        return cuda.event_elapsed_time(start_event, stop_event)


#### Tests — exercise 4 — scatter and gather

In [ ]:
def test4(h_input, destination_of, scatter_class=Scatter, gather_class=Gather):
    def check4(name, result, scattering):
        """The two are inverses of one another, and that is the whole point.

        scatter writes `destination[map(i)] = source[i]`, gather reads
        `destination[i] = source[map(i)]`. Building both oracles from the same
        index array is what keeps this from quietly checking one of them twice.
        """
        # The functor is a *device* function: the kernel hands it one thread
        # index at a time, so the oracle has to call it the same way rather
        # than pass it a whole array and hope it vectorises.
        where = np.fromiter((destination_of(i) for i in range(h_input.size)),
                            dtype=np.intp, count=h_input.size)
        if scattering:
            expected = np.empty_like(h_input)
            expected[where] = h_input
        else:
            expected = h_input[where]
        verify(name, result, expected)

    d_input = cuda.to_device(h_input)

    d_output = cuda.device_array_like(d_input)
    ct = scatter_class(destination_of)(d_input, d_output)
    check4("scatter", d_output.copy_to_host(), True)

    d_output = cuda.device_array_like(d_input)
    ct += gather_class(destination_of)(d_input, d_output)
    check4("gather", d_output.copy_to_host(), False)
    return ct


size = 1 << 16
middle = size // 2


def even_first(tid):
    """Send the even positions to the front and the odd ones to the back."""
    if (tid % 2) == 0:
        return tid // 2
    return middle + tid // 2


rng = np.random.default_rng(4)
test4(rng.integers(0, 1 << 20, size=size, dtype=np.int32), even_first)

## Exercise 5
Now that you can scan, writing the **histogram** pattern you must do.

The difficulty is that the output is tiny: a few hundred counters for millions of values, so every thread in the grid wants the same 256 addresses and a global `cuda.atomic.add` serialises the GPU onto them. The answer is **privatisation** — one copy of the histogram per block in *shared* memory, and a merge into the global one at the end.

Notice that the plumbing already does three things, so do not do them again:

- `d_histogram` arrives **already zeroed** (zeroing it in the kernel would erase counts another block has merged),
- the number of blocks is **capped** (`_MAX_BLOCK_NUMBER`) so that merges are few, so cover the input with a **grid-stride loop** as in labwork 2,
- `bin_op` is your binning function, already compiled as a device function.

In [ ]:
from __future__ import annotations

import numpy as np
from numba import cuda, core
from numba.np.numpy_support import from_dtype


def display(array, all_values=False):
    def aprint(data_range, noend=False):
        for i in data_range:
            print(f"{array[i]} ", end="")
            if (i % 16) == 15: print("")
        if not noend: print("...")

    if all_values:
        aprint(range(array.size), noend=True)
        return
    aprint(range(0, min(array.size, 16)))
    if array.size < 256 - 16: return
    aprint(range(256 - 16, min(array.size, 256 + 16)))
    if array.size < 512 - 16: return
    aprint(range(512 - 16, min(array.size, 512 + 16)))
    if array.size < 768 - 16: return
    aprint(range(768 - 16, min(array.size, 768 + 16)))
    aprint(range(array.size - 16 - 256, array.size - 256 + 16))
    aprint(range(array.size - 16, array.size), noend=True)


class Histogram(object):
    """Create a histogram object that counts values into bins, using a given
    unary function to say which bin a value falls in.

    The binning function is compiled once and cached inside this object.
    Keeping this object alive will prevent re-compilation.

    The whole point of this pattern is that the output is *tiny* -- a few
    hundred counters -- while the input is not.  Every thread in the grid wants
    to increment one of those few counters, so a naive kernel serialises the
    whole GPU onto 256 addresses in global memory.  The answer is privatisation:
    each block keeps its own copy of the histogram in shared memory, and only
    the per-block copies are merged into global memory at the end.  That trades
    a few million global atomics for a few thousand.
    """

    _kernels_histogram = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8
    _MAX_BLOCK_NUMBER = 84  # should be around 4 * number of SMP...

    @staticmethod
    def _gpu_kernel_factory(fn, histo_size, np_type, count_type):
        """Factory of kernels for the histogram problem...

        This function returns a Cuda Kernel that computes an histogram of some data."""

        bin_op = cuda.jit(func_or_sig=fn, device=True, cache=True)

        def gpu_histogram(d_input, d_histogram):
            """
            Histogram...

            `d_histogram` arrives already zeroed -- see `__call__`.  Zeroing it
            here would be a race: this grid has several blocks, and a block
            still writing zeros would erase the counts another block has
            already merged.
            """
            # TODO

        return cuda.jit(func_or_sig=gpu_histogram, cache=True)

    def __init__(self, functor, size):
        """
        :param functor: A function taking a value and returning the index of
                        the bin it belongs to.  It will be compiled as a CUDA
                        device function using ``cuda.jit(device=True)``.
        :param size: The number of bins.
        """
        self._functor = functor
        self._histo_size = size

    def _compile_kernel(self, np_type, count_type):
        key = self._functor, self._histo_size, np_type, count_type
        if key not in self._kernels_histogram:
            self._kernels_histogram[key] = Histogram._gpu_kernel_factory(
                self._functor, self._histo_size, from_dtype(np_type), from_dtype(count_type))
        return self._kernels_histogram[key]

    def __call__(self, d_input, d_histogram, stream=cuda.default_stream()):
        """ Performs a histogram calculation.

        :param d_input: A device array with input data.
        :param d_histogram: A device array of `size` counters to store the result.
        :param stream: Optional CUDA stream in which to perform the histogram.
                    If no stream is specified, the default stream of 0 is used.
        """

        # ensure 1d arrays
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # ensure the output has exactly the number of bins this object counts into
        if d_histogram.size != self._histo_size:
            raise ValueError(
                f"the histogram has {d_histogram.size} bins, expected {self._histo_size}"
            )

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel = self._compile_kernel(d_input.dtype, d_histogram.dtype)

        max_block_size = Histogram._WARP_SIZE * Histogram._NUM_WARPS
        nb_threads = min(max_block_size, d_input.size)
        nb_blocks = (d_input.size + nb_threads - 1) // nb_threads

        # We limit the number of blocks to use more atomics onto shared memory!
        # Each block costs `histo_size` global atomics at the end, so fewer,
        # longer-lived blocks mean fewer merges.  The grid-stride loop in the
        # kernel is what lets a fixed-size grid cover an input of any size.
        nb_blocks = min(nb_blocks, Histogram._MAX_BLOCK_NUMBER)

        start_event = cuda.event(True)
        start_event.record(stream=stream)

        # The counters must start at zero, and the kernel cannot do it itself.
        cuda.to_device(np.zeros(self._histo_size, dtype=d_histogram.dtype),
                       to=d_histogram, stream=stream)
        kernel[nb_blocks, nb_threads, stream](d_input, d_histogram)

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return cuda.event_elapsed_time(start_event, stop_event)

#### Tests — exercise 5 — the histogram

In [ ]:
def test5(h_input, n_bins, bin_of, histogram_class=Histogram):
    def check5(result):
        """The oracle: `bin_of` applied on the host, then numpy's count."""
        bins = np.asarray([bin_of(value) for value in h_input])
        expected = np.bincount(bins, minlength=n_bins)
        verify(f"histogram of {h_input.size} values into {n_bins} bins",
               result, expected)

    d_input = cuda.to_device(h_input)
    d_histogram = cuda.device_array(shape=n_bins, dtype=np.int32)
    histogram = histogram_class(bin_of, n_bins)

    ct = histogram(d_input, d_histogram)
    first = d_histogram.copy_to_host()
    check5(first)

    # Twice into the same array: a histogram that forgets to zero doubles.
    ct += histogram(d_input, d_histogram)
    verify("the same histogram, computed twice into the same array",
           d_histogram.copy_to_host(), first)
    return ct


rng = np.random.default_rng(5)

# Every bin used, and the identity binning: the simplest thing that can fail.
test5(rng.integers(0, 256, size=1 << 16, dtype=np.int32), 256, lambda a: a % 256)

# A coarser binning, so that the functor does real work, and a size that is not
# a multiple of the block size.
test5(rng.integers(0, 256, size=(1 << 16) + 37, dtype=np.int32), 32, lambda a: a // 8)

# A skewed input: one bin takes almost everything, which is the case where the
# atomics collide hardest.
skewed = rng.integers(0, 256, size=1 << 16, dtype=np.int32) // 128
test5(skewed, 2, lambda a: a)

#### Benchmark — the histogram

In [ ]:
def build(size):
    h_input = np.random.default_rng(size).integers(
        0, 256, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    d_histogram = cuda.device_array(shape=256, dtype=np.int32)
    histogram = Histogram(lambda a: a % 256, 256)
    # The traffic is one read of the input; the output is 1 KiB.
    return ((lambda: histogram(d_input, d_histogram)),
            (lambda: np.histogram(h_input, bins=256, range=(0, 256))),
            h_input.nbytes)


benchmark("histogram: GPU against numpy.histogram",
          build, (1 << 20, 1 << 22, 1 << 24))

## Exercise 6
To conclude, the **quantiles** of a column you must compute — and for this one you write almost no kernel:

| step | what it does | whose class |
|:--|:--|:--|
| 1 | value → the bin it falls in | `UnaryMap` — exercise 3 |
| 2 | count the bins | `Histogram` — exercise 5 |
| 3 | counts → how many values lie *below* each bin | `ExclusiveScan` — exercise 1 |
| 4 | counts → how many lie *at or above* each bin | `ReverseIScan` — exercise 2 |
| 5 | find the bin a probability lands in | one small kernel, and it is yours |

Step 3 answers *what value is the median*, step 4 answers *how many samples reach it*.

Only step 5 is new, because the index it reads is **searched for at run time**, which a map's functor cannot express. Given `d_cumulative`, find the **last** bin whose running total is still below `probability × n_values`, and report that bin's lower edge. Write it in two phases:

- **halve** the window while it is wider than a warp (three times for 256 bins). Each halving is a *dependent* read: the next cannot start before the last returns.
- then **let the warp finish it**: one bin per lane, each lane reading its own cumulative count and its neighbour's. Exactly one lane finds its own below the target and the next not; that lane writes.

So the kernel launches **one warp per probability**, not one thread. Hence `which` must be the same for every lane of a warp, and the block size a whole number of warps.

Notice also that the bounds are fixed when the object is built, since they are baked into the compiled binning function — keep one `Quantiles` alive rather than rebuilding it. And the binning **clips**: a value above `high` lands in the last bin, because it is still above the p99.

In [ ]:
from __future__ import annotations

import numpy as np
from numba import cuda, core
from numba.np.numpy_support import from_dtype


def display(array, all_values=False):
    def aprint(data_range, noend=False):
        for i in data_range:
            print(f"{array[i]} ", end="")
            if (i % 16) == 15: print("")
        if not noend: print("...")

    if all_values:
        aprint(range(array.size), noend=True)
        return
    aprint(range(0, min(array.size, 16)))
    if array.size < 256 - 16: return
    aprint(range(256 - 16, min(array.size, 256 + 16)))
    if array.size < 512 - 16: return
    aprint(range(512 - 16, min(array.size, 512 + 16)))
    if array.size < 768 - 16: return
    aprint(range(768 - 16, min(array.size, 768 + 16)))
    aprint(range(array.size - 16 - 256, array.size - 256 + 16))
    aprint(range(array.size - 16, array.size), noend=True)


class Quantiles(object):
    """The quantiles of a column of numbers, without sorting it.

    This exercise writes almost no kernel.  It is the assembly of the five
    before it, and that is the point: every pattern of this labwork is one step
    of the same pipeline.

    ====  ====================================  ==========================
    step  what it does                          whose class does it
    ====  ====================================  ==========================
    1     value -> the bin it falls in          `UnaryMap`      (exercise 3)
    2     count the bins                        `Histogram`     (exercise 5)
    3     counts -> how many lie below a bin    `ExclusiveScan` (exercise 1)
    4     counts -> how many lie at or above    `ReverseIScan`  (exercise 2)
    5     find the bin a probability lands in   one small kernel, below
    ====  ====================================  ==========================

    Steps 3 and 4 are the cumulative distribution and its complement: step 3
    answers "what value is the median", step 4 answers "how many samples exceed
    it", and a report wants both.

    Only step 5 is new, and it is new because it cannot be anything else: the
    index it reads is *searched for* at run time, which is exactly what the
    functor of a map or a gather cannot express.

    The bounds are fixed when the object is built, not passed to `__call__`,
    because they are baked into the compiled binning function.  Keeping one
    object alive for one column -- or for a set of columns that share a range --
    is what prevents a re-compilation on every call.
    """

    _kernels_search = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_search_factory(n_bins):
        """Factory of kernels for the quantile search...

        This function returns a Cuda Kernel with **one warp per probability**, in
        two phases.

        A binary search is the obvious way to find the bin, and it is the right
        way only for as long as the window is wider than a warp.  Once it is 32
        bins or fewer, a further halving costs one more dependent read to
        discard half of something the warp could have examined all at once -- so
        the search stops there, and the 32 lanes test the 32 remaining
        candidates in parallel, two reads each.  Exactly one of them holds the
        answer, and only that one writes.

        For 256 bins that is 3 halvings and one cooperative step, where a thread
        on its own would have done 8 dependent halvings."""

        WARP = Quantiles._WARP_SIZE

        def gpu_search(d_cumulative, d_exceeding, d_probabilities,
                       d_result, d_count_above, low, high, n_values):
            """
            Quantile lookup...
            """
            # TODO

        return cuda.jit(func_or_sig=gpu_search, cache=True)

    def __init__(self, low, high, n_bins=256):
        """
        :param low: The lower edge of the first bin.
        :param high: The upper edge of the last bin.
        :param n_bins: The number of bins, which is also the resolution of the
                       answer: a quantile is reported as the lower edge of the
                       bin it falls in.
        """
        if high <= low:
            raise ValueError(f"the range is empty ({low} / {high})")
        self._low = float(low)
        self._high = float(high)
        self._n_bins = int(n_bins)

        low, high, n_bins = self._low, self._high, self._n_bins

        def bin_of(value):
            """Where a value falls, clamped so that `high` lands in the last bin."""
            index = int((value - low) / (high - low) * n_bins)
            if index < 0:
                index = 0
            if index >= n_bins:
                index = n_bins - 1
            return index

        # The five patterns of this labwork, built once and kept: each of these
        # objects caches a compiled kernel, and throwing them away between calls
        # throws the compilation away with them.
        from_exercise3 = UnaryMap(bin_of)
        from_exercise5 = Histogram(lambda a: a, n_bins)
        from_exercise1 = ExclusiveScan(lambda a, b: a + b)
        from_exercise2 = ReverseIScan(lambda a, b: a + b)

        self._to_bins = from_exercise3
        self._count = from_exercise5
        self._below = from_exercise1
        self._at_or_above = from_exercise2

    def _compile_search(self):
        key = self._n_bins
        if key not in self._kernels_search:
            self._kernels_search[key] = Quantiles._gpu_kernel_search_factory(self._n_bins)
        return self._kernels_search[key]

    def __call__(self, d_column, d_probabilities, d_result, d_count_above,
                 stream=cuda.default_stream()):
        """ Computes the quantiles of one column.

        :param d_column: A device array of values.
        :param d_probabilities: A device array of the probabilities wanted, each
                                in [0, 1] -- 0.5 for the median.
        :param d_result: A device array of `d_probabilities.size` values, to
                         receive the quantiles.
        :param d_count_above: A device array of `d_probabilities.size` counters,
                              to receive how many values reach each quantile.
        :param stream: Optional CUDA stream in which to work.
                    If no stream is specified, the default stream of 0 is used.
        :return: a tuple (map, histogram, scan, reverse scan, search) of times in ms.
        """

        # ensure 1d array
        if d_column.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_column.size < 1:
            raise ValueError("array's length is 0")

        # ensure there is room for one answer per probability
        if d_result.size != d_probabilities.size or d_count_above.size != d_probabilities.size:
            raise ValueError(
                f"{d_probabilities.size} probabilities but room for "
                f"{d_result.size} / {d_count_above.size} answers"
            )

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        n_bins = self._n_bins
        bins = cuda.device_array(shape=d_column.size, dtype=np.int32)
        counts = cuda.device_array(shape=n_bins, dtype=np.int32)
        below = cuda.device_array(shape=n_bins, dtype=np.int32)
        at_or_above = cuda.device_array(shape=n_bins, dtype=np.int32)

        # Steps 1 to 4 are the exercises above, called and not rewritten.
        ct_map = self._to_bins(d_column, bins, stream)
        ct_histogram = self._count(bins, counts, stream)
        ct_below = self._below(counts, below, np.int32(0), stream)
        ct_above = self._at_or_above(counts, at_or_above, stream)

        kernel = self._compile_search()
        # One warp per probability, so the block size must stay a whole number
        # of warps -- a half-warp block would split a search across two of them.
        max_block_size = Quantiles._WARP_SIZE * Quantiles._NUM_WARPS
        wanted = d_probabilities.size * Quantiles._WARP_SIZE
        nb_threads = min(max_block_size, ((wanted + Quantiles._WARP_SIZE - 1)
                                          // Quantiles._WARP_SIZE) * Quantiles._WARP_SIZE)
        nb_blocks = (wanted + nb_threads - 1) // nb_threads

        start_event = cuda.event(True)
        start_event.record(stream=stream)
        kernel[nb_blocks, nb_threads, stream](
            below, at_or_above, d_probabilities, d_result, d_count_above,
            self._low, self._high, d_column.size)
        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return (ct_map, ct_histogram, ct_below, ct_above,
                cuda.event_elapsed_time(start_event, stop_event))

#### Tests — exercise 6 — the quantiles

In [ ]:
def test6(h_column, low, high, h_probabilities, n_bins=256, quantiles_class=Quantiles):
    def check6(values, reached):
        """The same answer, sequentially, with numpy.

        Histogram-based, since the kernel answers with a bin's lower edge, and
        clipping rather than dropping what falls outside the range.
        """
        indices = np.clip(((h_column - low) / (high - low) * n_bins).astype(np.int64),
                          0, n_bins - 1)
        counts = np.bincount(indices, minlength=n_bins)
        below = np.concatenate(([0], np.cumsum(counts)[:-1]))
        at_or_above = np.flip(np.cumsum(np.flip(counts)))

        want_values = np.empty(h_probabilities.size, dtype=np.float32)
        want_reached = np.empty(h_probabilities.size, dtype=np.int32)
        for k, probability in enumerate(h_probabilities):
            target = float(probability) * h_column.size
            found = int(np.searchsorted(below, target, side="left")) - 1
            found = min(max(found, 0), n_bins - 1)
            want_values[k] = low + (high - low) * found / n_bins
            want_reached[k] = at_or_above[found]

        name = f"quantiles of {h_column.size} values in {n_bins} bins"
        verify(name, values, want_values, tolerance=1e-5)
        verify(name + ", and how many values reach them", reached, want_reached)

    d_result = cuda.device_array(shape=h_probabilities.size, dtype=np.float32)
    d_count_above = cuda.device_array(shape=h_probabilities.size, dtype=np.int32)
    ct = quantiles_class(low, high, n_bins)(
        cuda.to_device(h_column), cuda.to_device(h_probabilities),
        d_result, d_count_above)
    check6(d_result.copy_to_host(), d_count_above.copy_to_host())
    return sum(ct)


rng = np.random.default_rng(6)
probabilities = np.array([0.0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0], dtype=np.float32)

# The ordinary case, and a size that is not a multiple of the block size.
test6(rng.normal(size=1 << 16).astype(np.float32), -6.0, 6.0, probabilities)
test6(rng.normal(size=(1 << 14) + 13).astype(np.float32), -6.0, 6.0, probabilities)

# A heavy tail, so that clipping is exercised rather than merely present: a
# Cauchy sample leaves the range on both sides and none of it may be lost.
test6(rng.standard_cauchy(size=1 << 14).astype(np.float32), -6.0, 6.0, probabilities)

# Degenerate columns, which is where an off-by-one in the search shows up.
test6(np.full(1 << 12, 1.25, dtype=np.float32), -6.0, 6.0, probabilities)
test6(np.array([0.7], dtype=np.float32), -6.0, 6.0, probabilities)

# A coarse bin count: the answer gets worse, the code must not.
test6(rng.normal(size=1 << 14).astype(np.float32), -6.0, 6.0, probabilities, n_bins=32)

#### Benchmark — the quantiles

In [ ]:
def sweep(sizes=(1 << 20, 1 << 22, 1 << 24), n_bins=256, repeats=10):
    """The five steps, timed apart -- which is the whole point of the exercise."""
    rng = np.random.default_rng(6)
    probabilities = np.array([0.01, 0.25, 0.5, 0.75, 0.99], dtype=np.float32)
    d_probabilities = cuda.to_device(probabilities)
    quantiles = Quantiles(-6.0, 6.0, n_bins)

    print(f"one column, {n_bins} bins, times in ms")
    print(f"{'values':>10} | {'1 map':>7} | {'2 histo':>7} | {'3 scan':>7} | "
          f"{'4 reverse':>9} | {'5 search':>8} | {'numpy':>8}")
    print("-" * 76)
    for size in sizes:
        h_column = rng.normal(size=size).astype(np.float32)
        d_column = cuda.to_device(h_column)
        d_result = cuda.device_array(shape=probabilities.size, dtype=np.float32)
        d_above = cuda.device_array(shape=probabilities.size, dtype=np.int32)
        call = lambda: quantiles(d_column, d_probabilities, d_result, d_above)
        for _ in range(3):
            call()
        cuda.synchronize()
        best = min((call() for _ in range(repeats)), key=sum)
        reference = min(_time_once(
            lambda: np.quantile(h_column, probabilities)) for _ in range(3))
        print(f"{size:10d} | {best[0]:7.3f} | {best[1]:7.3f} | {best[2]:7.3f} | "
              f"{best[3]:9.3f} | {best[4]:8.3f} | {reference:8.3f}")
    print()
    print("Columns 1 and 2 grow with the data. Columns 3 and 4 do not: they are")
    print("256 numbers whatever the input, and what you are timing there is the")
    print("launch, not the addition.")


def searching(size=1 << 24, n_bins=256, repeats=20):
    """Step 5 alone, against the number of quantiles asked for.

    Five probabilities is five warps: the kernel is a launch and nothing else,
    and the two-phase search cannot show up. Ask for a quantile per bin -- which
    is what equal-frequency binning does -- and there is finally work to measure.
    """
    rng = np.random.default_rng(6)
    h_column = rng.normal(size=size).astype(np.float32)
    d_column = cuda.to_device(h_column)
    quantiles = Quantiles(-6.0, 6.0, n_bins)

    print()
    print(f"step 5 alone, {n_bins} bins, {size} values, times in ms")
    print(f"{'quantiles':>10} | {'warps':>7} | {'search (ms)':>12} | {'per quantile':>13}")
    print("-" * 52)
    for count in (5, 32, 256, 4096):
        probabilities = np.linspace(0.0, 1.0, count, dtype=np.float32)
        d_probabilities = cuda.to_device(probabilities)
        d_result = cuda.device_array(shape=count, dtype=np.float32)
        d_above = cuda.device_array(shape=count, dtype=np.int32)
        call = lambda: quantiles(d_column, d_probabilities, d_result, d_above)
        for _ in range(3):
            call()
        cuda.synchronize()
        best = min(call()[4] for _ in range(repeats))
        print(f"{count:10d} | {count:7d} | {best:12.4f} | {best / count * 1000:12.4f}us")
    print()
    print("The per-quantile cost falls as the launch is amortised. Three")
    print("dependent reads and one cooperative warp step is what each of those")
    print("searches costs; a lone thread would have done eight dependent reads.")


sweep()
searching()

### Seeing what exercise 6 computed

A quantile is a place on a curve, and so far you have only compared it with
NumPy. The cell below draws the two arrays the pipeline builds: the histogram of
exercise 5, and the cumulative distribution of exercise 1. Each dashed line is
one probability, read across to the curve and down to the value — which is the
search of step 5, drawn.

In [ ]:
import matplotlib.pyplot as plt

LOW, HIGH, N_BINS = -6.0, 6.0, 256
STEP = (HIGH - LOW) / N_BINS
column = np.random.default_rng(6).normal(size=1 << 16).astype(np.float32)
probabilities = np.array([0.01, 0.25, 0.5, 0.75, 0.99], dtype=np.float32)

d_column = cuda.to_device(column)
d_values = cuda.device_array(probabilities.size, dtype=np.float32)
d_reached = cuda.device_array(probabilities.size, dtype=np.int32)
Quantiles(LOW, HIGH, N_BINS)(d_column, cuda.to_device(probabilities),
                             d_values, d_reached)
values = d_values.copy_to_host()

# The first three steps again, kept this time instead of thrown away, so that
# they can be drawn.
def bin_of(value):
    index = int((value - LOW) / (HIGH - LOW) * N_BINS)
    if index < 0:
        index = 0
    if index >= N_BINS:
        index = N_BINS - 1
    return index

d_bins = cuda.device_array(column.size, dtype=np.int32)
d_histogram = cuda.device_array(N_BINS, dtype=np.int32)
d_cumulative = cuda.device_array(N_BINS, dtype=np.int32)
UnaryMap(bin_of)(d_column, d_bins)
Histogram(lambda a: a, N_BINS)(d_bins, d_histogram)
ExclusiveScan(lambda a, b: a + b)(d_histogram, d_cumulative, np.int32(0))

histogram = d_histogram.copy_to_host()
cumulative = d_cumulative.copy_to_host()
edges = LOW + STEP * np.arange(N_BINS)

figure, (left, right) = plt.subplots(1, 2, figsize=(9, 3.2))
left.bar(edges, histogram, width=STEP, color="0.8")
for value in values:
    left.axvline(value, color="crimson", linewidth=0.8)
left.set_title("histogram (exercise 5)")
left.set_xlabel("value")
left.set_xlim(-4.5, 4.5)

right.plot(edges, cumulative / column.size, linewidth=1.2)
for probability, value in zip(probabilities, values):
    right.plot([LOW, value], [probability, probability],
               color="crimson", linewidth=0.6, linestyle="--")
    right.plot([value, value], [0, probability],
               color="crimson", linewidth=0.6, linestyle="--")
right.set_title("cumulative distribution (exercise 1)")
right.set_xlabel("value")
right.set_ylabel("proportion below")
right.set_xlim(-4.5, 4.5)
right.set_ylim(0, 1)
figure.tight_layout()
plt.show()

for probability, value, reached in zip(probabilities, values,
                                       d_reached.copy_to_host()):
    print(f"p{probability:<5.2f} = {value:7.3f}   reached by {reached:6d} values")

## Afterwards — what it was all for

The table below is real: the districts of the 1990 US census that Colab ships in
`/content/sample_data`. Its columns are a longitude, a count of rooms and a house
price, so they share no scale, and some of them have the long right tail that
real measurements have and generated ones do not.

Every step is one of your own classes, and nothing leaves the GPU between them:

| | |
|:--|:--|
| the table arrives sample-major, the pipeline needs feature-major | **`Scatter`** — exercise 4, since a transpose is a permutation |
| each column's p1, quartiles, median and p99 | **`Quantiles`** — exercise 6, and through it 1, 2, 3 and 5 |
| clip to [p1, p99] and rescale by the median and the IQR | **`UnaryMap`** — exercise 3 |

That is scikit-learn's `RobustScaler`. The first table printed below is the
reason it is built on a median and an interquartile range: compare each column's
mean with its median, and its standard deviation with its IQR.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

frame = pd.read_csv("/content/sample_data/california_housing_train.csv")
names = list(frame.columns)
h_table = frame.to_numpy(dtype=np.float32)
rows, columns = h_table.shape
print(f"{rows} districts, {columns} measurements each, 1990 US census")

# --- 1. transpose, so that a column is contiguous -- exercise 4 --------------
def to_feature_major(tid):
    # Sample-major index -> feature-major index. A pure permutation.
    return (tid % columns) * rows + tid // columns


d_flat = cuda.to_device(h_table.reshape(-1))
d_major = cuda.device_array(shape=d_flat.size, dtype=np.float32)
ct = Scatter(to_feature_major)(d_flat, d_major)
print(f"transpose: {ct:.3f} ms")

# --- 2. the five quantiles of every column -- exercise 6 ---------------------
probabilities = np.array([0.01, 0.25, 0.5, 0.75, 0.99], dtype=np.float32)
d_probabilities = cuda.to_device(probabilities)

# The range of each column, which is all the host is asked for.
lows = h_table.min(axis=0)
highs = h_table.max(axis=0)

summary = np.empty((columns, probabilities.size), dtype=np.float32)
total = 0.0
for column in range(columns):
    values = d_major[column * rows:(column + 1) * rows]
    d_result = cuda.device_array(shape=probabilities.size, dtype=np.float32)
    d_above = cuda.device_array(shape=probabilities.size, dtype=np.int32)
    total += sum(Quantiles(float(lows[column]), float(highs[column]), 256)(
        values, d_probabilities, d_result, d_above))
    summary[column] = d_result.copy_to_host()
print(f"quantiles of {columns} columns: {total:.3f} ms")

# --- 3. clip and rescale -- exercise 3 ---------------------------------------
d_scaled = cuda.device_array(shape=d_major.size, dtype=np.float32)
total = 0.0
for column in range(columns):
    p1, q1, median, q3, p99 = (float(v) for v in summary[column])
    spread = q3 - q1 if q3 > q1 else 1.0

    def robust(value, p1=p1, p99=p99, median=median, spread=spread):
        if value < p1:
            value = p1
        if value > p99:
            value = p99
        return (value - median) / spread

    source = d_major[column * rows:(column + 1) * rows]
    target = d_scaled[column * rows:(column + 1) * rows]
    # UnaryMap returns (destination, ct), not just the time.
    _, ct = UnaryMap(robust)(source, target)
    total += ct
print(f"clip and rescale: {total:.3f} ms")

# --- and what it bought ------------------------------------------------------
after = d_scaled.copy_to_host().reshape(columns, rows)

print()
print(f"{'feature':>20} | {'mean':>11} | {'median':>11} | {'std':>11} | "
      f"{'IQR':>11} | {'tail':>6}")
print("-" * 84)
tails = np.empty(columns)
for column in range(columns):
    raw = h_table[:, column]
    p1, q1, median, q3, p99 = (float(v) for v in summary[column])
    # How far the largest value lies beyond the bulk, in interquartile ranges.
    tails[column] = (raw.max() - p99) / (q3 - q1) if q3 > q1 else 0.0
    print(f"{names[column][:20]:>20} | {raw.mean():11.2f} | {median:11.2f} | "
          f"{raw.std():11.2f} | {q3 - q1:11.2f} | {tails[column]:5.1f}x")
print()
print("Where the mean sits far from the median, and the standard deviation far")
print("from the IQR, the column is skewed: the last column says how far past the")
print("p99 its largest value reaches, in interquartile ranges.")

print()
print(f"{'feature':>20} | {'scaled mean':>12} | {'scaled std':>11} | "
      f"{'scaled min':>11} | {'scaled max':>11}")
print("-" * 76)
for column in range(columns):
    done = after[column]
    print(f"{names[column][:20]:>20} | {done.mean():12.3f} | {done.std():11.3f} | "
          f"{done.min():11.3f} | {done.max():11.3f}")
print()
print("Every column now sits on the same scale, centred near zero, with no value")
print("further out than the clip allowed -- which is what a model wants.")

# The most skewed column, before and after.
worst = int(np.argmax(tails))
figure, (left, right) = plt.subplots(1, 2, figsize=(9, 3))
left.hist(h_table[:, worst], bins=120, color="0.7")
for value in summary[worst][[0, 2, 4]]:
    left.axvline(value, color="crimson", linewidth=0.8)
left.set_title(f"{names[worst]}, as recorded")
left.set_yscale("log")
right.hist(after[worst], bins=120, color="0.7")
right.set_title("clipped to [p1, p99] and rescaled")
figure.tight_layout()
plt.show()

## A real problem — equalising a photograph

The same three patterns solve a problem nobody would call statistics.
**Histogram equalisation** replaces each grey level by the proportion of pixels
below it, which stretches crowded levels apart and squeezes empty ones together.
That proportion is the cumulative distribution of exercise 1, and the
replacement is the map of exercise 3.

Done plainly it overreacts. One level here holds eight times an average bin —
the dark uniform — so the table climbs eight levels at a step just there, and
the noise already in the JPEG is magnified into blotches. Clipping every bin to
`clip_value` times the average before the scan, and sharing the removed pixels
out equally, bounds that step: the picture gives up a little of its perfect
uniformity and keeps its blacks.

The image ships with matplotlib, so nothing is downloaded. Only the 256 counts
and the 256-entry table cross the bus; the pixels stay on the card.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook


def equalisation_tables(flat, clip_value=2):
    """Count, clip and scan one image: its plain and contrast-limited tables.

    The clip is the only difference between the two. No bin may exceed
    `clip_value` times the average, and the pixels cut off the peaks are shared
    equally over the 256 bins -- a correction 256 numbers wide, which is why it
    sits on the host.
    """
    d_histogram = cuda.device_array(256, dtype=np.int32)
    ct_count = Histogram(lambda a: a, 256)(cuda.to_device(flat),
                                           d_histogram)               # exercise 5
    counts = d_histogram.copy_to_host()

    def lookup_table(bins):
        """Counts to the 256-entry map that flattens them."""
        d_cumulative = cuda.device_array(256, dtype=np.int32)
        ct = ExclusiveScan(lambda a, b: a + b)(cuda.to_device(bins), d_cumulative,
                                               np.int32(0))           # exercise 1
        cumulative = d_cumulative.copy_to_host()
        floor = int(cumulative[cumulative > 0][0])
        table = np.round((cumulative - floor) / (int(bins.sum()) - floor) * 255)
        return np.clip(table, 0, 255).astype(np.int32), cumulative, ct

    limit = int(clip_value * flat.size // 256)
    clipped = (np.minimum(counts, limit)
               + int(np.maximum(counts - limit, 0).sum()) // 256).astype(np.int32)
    plain, cumulative, ct_scan = lookup_table(counts)
    tamed, _, _ = lookup_table(clipped)
    return counts, limit, cumulative, plain, tamed, (ct_count, ct_scan)


def show_equalisation(images, counts, limit, cumulative, tables, log=False):
    """The three pictures, with the three curves that explain them below.

    The original sits between its two corrections, so each is read against it.
    """
    as_recorded, plain_image, tamed_image = images
    plain, tamed = tables
    figure, axes = plt.subplots(2, 3, figsize=(12, 7.5))
    for ax, image, title in zip(axes[0], (plain_image, as_recorded, tamed_image),
                                ("equalised", "as recorded", "equalised, clipped")):
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
    axes[1, 0].bar(np.arange(256), counts, width=1, color="0.7")
    axes[1, 0].axhline(limit, color="crimson", linewidth=1)
    if log:
        axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("histogram, with the clip limit (exercise 5)")
    axes[1, 1].plot(plain, label="plain")
    axes[1, 1].plot(tamed, label="clipped")
    axes[1, 1].plot([0, 255], [0, 255], "k--", linewidth=0.6)
    axes[1, 1].set_title("the lookup table")
    axes[1, 1].legend(fontsize=7)
    total = as_recorded.size
    axes[1, 2].plot(cumulative / total, label="before")
    for image, label in ((plain_image, "plain"), (tamed_image, "clipped")):
        axes[1, 2].plot(np.cumsum(np.bincount(image.ravel(), minlength=256)) / total,
                        label=label)
    axes[1, 2].plot([0, 255], [0, 1], "k--", linewidth=0.6)
    axes[1, 2].set_title("cumulative distribution (exercise 1)")
    axes[1, 2].legend(fontsize=7)
    figure.tight_layout()
    plt.show()


with cbook.get_sample_data("grace_hopper.jpg") as handle:
    rgb = plt.imread(handle)
grey = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1]
        + 0.114 * rgb[..., 2]).astype(np.int32)
flat = grey.ravel()
print(f"{grey.shape[1]} x {grey.shape[0]} pixels, {flat.size} grey levels to count")

(counts, limit, cumulative, photo_plain, photo_tamed,
 (ct_count, ct_scan)) = equalisation_tables(flat, clip_value=2)

d_image = cuda.to_device(flat)
d_plain, ct_map = UnaryMap(lambda level: photo_plain[level])(d_image)  # exercise 3
d_tamed, _ = UnaryMap(lambda level: photo_tamed[level])(d_image)
plain_image = d_plain.copy_to_host().reshape(grey.shape)
tamed_image = d_tamed.copy_to_host().reshape(grey.shape)

# First call to each class, so these are compilation times more than run times.
print(f"count {ct_count:.1f} ms, scan {ct_scan:.1f} ms, map {ct_map:.1f} ms")
print(f"contrast (standard deviation of the grey levels): {grey.std():.1f} as "
      f"recorded, {plain_image.std():.1f} plain, {tamed_image.std():.1f} clipped")
print(f"steepest step in the table: {np.diff(photo_plain).max()} levels plain, "
      f"{np.diff(photo_tamed).max()} clipped")

show_equalisation((grey, plain_image, tamed_image), counts, limit, cumulative,
                  (photo_plain, photo_tamed))

### The same tables, a harder histogram

A scanner records more grey levels than a screen can show, and most of a slice
is background. This one is 256 x 256 in 16 bits, rescaled to 256 levels before
being counted.

Its worst bin is far worse than the photograph's — 145 times the average against
eight — but it sits at level 0, and the floor the table subtracts already sends
that level to black. The plain table climbs two levels at a step here rather
than eight: there was nothing to repair. Tighten the limit to 1.25 and still
only that one bin exceeds it; the clip's only real effect is to spread the
background over the other 255 bins, which drags the table back towards the
diagonal and costs contrast.

So the limit is not free insurance. It answers a peak among the levels you are
keeping, and this image has none.

In [ ]:
with cbook.get_sample_data("s1045.ima.gz") as handle:
    raw = np.frombuffer(handle.read(), np.uint16).reshape(256, 256)
scan_flat = np.round(raw / raw.max() * 255).astype(np.int32).ravel()

(scan_counts, scan_limit, scan_cumulative, scan_plain, scan_tamed,
 _) = equalisation_tables(scan_flat, clip_value=1.25)

d_scan = cuda.to_device(scan_flat)
d_scan_plain, _ = UnaryMap(lambda level: scan_plain[level])(d_scan)    # exercise 3
d_scan_tamed, _ = UnaryMap(lambda level: scan_tamed[level])(d_scan)
as_recorded = scan_flat.reshape(256, 256)
plain_scan = d_scan_plain.copy_to_host().reshape(256, 256)
tamed_scan = d_scan_tamed.copy_to_host().reshape(256, 256)

print(f"worst bin {scan_counts.max()} pixels at level {scan_counts.argmax()}, "
      f"{scan_counts.max() / (scan_flat.size / 256):.0f} times the average; "
      f"{int((scan_counts > scan_limit).sum())} bin above the limit")
print(f"contrast: {as_recorded.std():.1f} as recorded, {plain_scan.std():.1f} "
      f"plain, {tamed_scan.std():.1f} clipped")
print(f"steepest step in the table: {np.diff(scan_plain).max()} levels plain, "
      f"{np.diff(scan_tamed).max()} clipped")

show_equalisation((as_recorded, plain_scan, tamed_scan), scan_counts, scan_limit,
                  scan_cumulative, (scan_plain, scan_tamed), log=True)

### And an image that really wants it

Neither photograph nor scan needed much help. This one does: a 344 x 403
elevation model of the Jacksboro fault, ground height rescaled to 256 levels.
Most of the ground lies in the lower half of the altitude range, so as recorded
the map is dark and flat and the drainage network cut into it is hard to read.

Equalisation is exactly the right tool here, and its worst bin is only three
times the average, so the clip has nothing to bite on and the two corrections
agree. Same three kernels, same 256-entry table, third answer.

In [ ]:
with cbook.get_sample_data("jacksboro_fault_dem.npz") as handle:
    elevation = handle["elevation"].astype(np.float32)
terrain = np.round((elevation - elevation.min())
                   / (elevation.max() - elevation.min()) * 255).astype(np.int32)
terrain_flat = terrain.ravel()

(dem_counts, dem_limit, dem_cumulative, dem_plain, dem_tamed,
 _) = equalisation_tables(terrain_flat, clip_value=2)

d_terrain = cuda.to_device(terrain_flat)
d_dem_plain, _ = UnaryMap(lambda level: dem_plain[level])(d_terrain)   # exercise 3
d_dem_tamed, _ = UnaryMap(lambda level: dem_tamed[level])(d_terrain)
plain_terrain = d_dem_plain.copy_to_host().reshape(terrain.shape)
tamed_terrain = d_dem_tamed.copy_to_host().reshape(terrain.shape)

print(f"worst bin {dem_counts.max()} cells at level {dem_counts.argmax()}, "
      f"{dem_counts.max() / (terrain_flat.size / 256):.0f} times the average")
print(f"contrast: {terrain.std():.1f} as recorded, {plain_terrain.std():.1f} "
      f"plain, {tamed_terrain.std():.1f} clipped")

show_equalisation((terrain, plain_terrain, tamed_terrain), dem_counts, dem_limit,
                  dem_cumulative, (dem_plain, dem_tamed))